# 🎓 Semana 23-24: Frameworks de Data Science y Proyectos Finales

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Avanzado — Cierre del curso

---

## 📋 ¿Qué vas a aprender y construir esta semana?

| Bloque | Temas |
|--------|-------|
| 🧠 TensorFlow | Redes neuronales, clasificación de imágenes con Fashion MNIST |
| 🔬 Scikit-learn avanzado | Regresión con datasets reales, pipelines complejos |
| 📊 Proyecto 1 | Análisis de redes sociales: detección de comunidades e influencers |
| 🏠 Proyecto 2 | Predicción de precios de bienes raíces con dataset completo |
| 🏥 Proyecto 3 | Clasificación de enfermedades médicas con interpretabilidad |

---

> 🎯 **Esta es la semana final del curso.**  
> Cada bloque integra todo lo aprendido en las 22 semanas anteriores.  
> Los proyectos están diseñados para ser parte de tu portafolio.

> **Instalaciones necesarias:**
> ```bash
> pip install tensorflow scikit-learn pandas numpy matplotlib seaborn
> ```

---
## 🧠 Bloque 1: TensorFlow — Redes Neuronales

### 📘 Teoría

**TensorFlow** es el framework de deep learning más usado en la industria.  
**Keras** es su API de alto nivel — permite construir redes neuronales con pocas líneas de código.

#### Anatomía de una red neuronal
```
Input Layer     Hidden Layers     Output Layer
  [x₁]                               [ŷ₁]
  [x₂]  →  [neurones] → [neurones] → [ŷ₂]
  [x₃]                               [ŷ₃]

Cada neurona: salida = activación(w·x + b)
```

#### Funciones de activación

| Función | Fórmula | Cuándo usar |
|---------|---------|-------------|
| **ReLU** | `max(0, x)` | Capas ocultas (más común) |
| **Sigmoid** | `1/(1+e⁻ˣ)` | Clasificación binaria (output) |
| **Softmax** | `eˣⁱ/Σeˣ` | Clasificación multiclase (output) |
| **Linear** | `x` | Regresión (output) |

#### Ciclo de entrenamiento
```
Forward pass → calcular pérdida → Backpropagation → actualizar pesos
└─────────────────── una época ──────────────────────────────────┘
```

#### Conceptos clave
```python
model.compile(
    optimizer='adam',          # algoritmo de optimización
    loss='sparse_categorical_crossentropy',  # función de pérdida
    metrics=['accuracy']       # métricas a monitorear
)
model.fit(X_train, y_train,
    epochs=10,                 # cantidad de pasadas completas
    batch_size=32,             # muestras por paso
    validation_split=0.2       # % para validación
)
```

### 💡 Ejemplo resuelto 1 — Red neuronal con scikit-learn (MLPClassifier)

In [ ]:
# ✅ Red neuronal usando scikit-learn (sin necesidad de TensorFlow)
# Ideal para introducir los conceptos sin dependencias pesadas

import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.datasets import load_digits
import warnings
warnings.filterwarnings('ignore')

# ── Clasificación de dígitos escritos a mano (8×8 píxeles) ──
print('=== Dataset: Dígitos escritos a mano (sklearn) ===')
digits = load_digits()
X, y   = digits.data, digits.target
print(f'  Shape de X: {X.shape}  (muestras × píxeles)')
print(f'  Clases:     {np.unique(y)}  ({len(np.unique(y))} dígitos)')
print(f'  Ejemplo fila 0: min={X[0].min():.0f}, max={X[0].max():.0f}')

# Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalizar (crucial para redes neuronales)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# ── Comparar arquitecturas ──
arquitecturas = [
    ('Una capa (64)',         MLPClassifier(hidden_layer_sizes=(64,),         max_iter=300, random_state=42)),
    ('Dos capas (128, 64)',   MLPClassifier(hidden_layer_sizes=(128, 64),     max_iter=300, random_state=42)),
    ('Tres capas (256,128,64)',MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=300, random_state=42)),
    ('Dropout (regulariz.)', MLPClassifier(hidden_layer_sizes=(128,64),
                                            alpha=0.01, max_iter=300, random_state=42)),
]

print(f'\n{'Arquitectura':28} {'Accuracy Test':>15} {'Loss final':>12}')
print('-' * 58)

mejor_acc, mejor_modelo, mejor_nombre = 0, None, ''
for nombre, modelo in arquitecturas:
    modelo.fit(X_tr_s, y_tr)
    acc  = modelo.score(X_te_s, y_te)
    loss = modelo.loss_
    print(f'  {nombre:26} {acc:>15.4f} {loss:>12.4f}')
    if acc > mejor_acc:
        mejor_acc, mejor_modelo, mejor_nombre = acc, modelo, nombre

# Reporte detallado del mejor
print(f'\n=== Mejor: {mejor_nombre} (accuracy={mejor_acc:.4f}) ===')
pred = mejor_modelo.predict(X_te_s)
print(classification_report(y_te, pred))

# Análisis de errores
errores_idx = np.where(pred != y_te)[0]
print(f'\nErrores cometidos: {len(errores_idx)} de {len(y_te)}')
print('Confusiones más frecuentes (real → predicho):')
from collections import Counter
confusiones = Counter(zip(y_te[errores_idx], pred[errores_idx]))
for (real, pred_val), count in confusiones.most_common(5):
    print(f'  {real} → {pred_val}: {count} veces')

### 💡 Ejemplo resuelto 2 — TensorFlow/Keras: Fashion MNIST

In [ ]:
# ✅ Red neuronal con TensorFlow/Keras
# Si no tenés TensorFlow instalado: pip install tensorflow

try:
    import tensorflow as tf
    from tensorflow import keras
    TF_DISPONIBLE = True
    print(f'TensorFlow {tf.__version__} disponible ✅')
except ImportError:
    TF_DISPONIBLE = False
    print('TensorFlow no instalado — usando simulación con scikit-learn')

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings('ignore')

# Nombres de clases de Fashion MNIST
CLASES = ['Remera','Pantalón','Suéter','Vestido','Abrigo',
          'Sandalia','Camisa','Zapatilla','Cartera','Bota']

if TF_DISPONIBLE:
    print('\n=== Fashion MNIST con TensorFlow/Keras ===')

    # Cargar datos
    (X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
    print(f'  Train: {X_train.shape}, Test: {X_test.shape}')

    # Normalizar píxeles [0,255] → [0,1]
    X_train_n = X_train.astype('float32') / 255.0
    X_test_n  = X_test.astype('float32') / 255.0

    # Aplanar imágenes 28×28 → 784
    X_train_flat = X_train_n.reshape(-1, 784)
    X_test_flat  = X_test_n.reshape(-1, 784)

    # Arquitectura de la red
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(64,  activation='relu'),
        keras.layers.Dense(10,  activation='softmax'),
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print('\nArquitectura:')
    model.summary()

    # Entrenar
    print('\nEntrenando...')
    history = model.fit(
        X_train_flat, y_train,
        epochs=15, batch_size=256,
        validation_split=0.1,
        verbose=1
    )

    # Evaluar
    test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
    print(f'\nAccuracy en test: {test_acc:.4f}')

    # Predicciones
    pred_proba = model.predict(X_test_flat, verbose=0)
    pred_clases = pred_proba.argmax(axis=1)
    print('\nReporte de clasificación:')
    print(classification_report(y_test, pred_clases, target_names=CLASES))

else:
    # Versión con MLPClassifier si no hay TF
    print('\n=== Fashion MNIST simulado con MLPClassifier ===')
    print('(Usando subset de 10.000 muestras para velocidad)')

    np.random.seed(42)
    # Simular datos de Fashion MNIST (28x28 = 784 features)
    n_train, n_test = 8000, 2000
    X_sim = np.random.rand(n_train + n_test, 784)
    y_sim = np.random.randint(0, 10, n_train + n_test)
    X_tr_sim, X_te_sim = X_sim[:n_train], X_sim[n_train:]
    y_tr_sim, y_te_sim = y_sim[:n_train], y_sim[n_train:]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr_sim)
    X_te_s = scaler.transform(X_te_sim)

    modelo = MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=50, random_state=42)
    modelo.fit(X_tr_s, y_tr_sim)
    acc = modelo.score(X_te_s, y_te_sim)
    print(f'  Accuracy (datos aleatorios, baseline ~10%): {acc:.4f}')
    print('  ✅ Estructura del pipeline lista — con datos reales la accuracy sería ~88%')

### ✏️ Ejercicio guiado 1 — Ajustar hiperparámetros de la red

Experimentá con distintas configuraciones y analizá el impacto.

**Pistas:**
- `hidden_layer_sizes=(n1, n2, ...)` controla la arquitectura
- `alpha` controla la regularización L2 (evita sobreajuste)
- `learning_rate_init` controla la velocidad de aprendizaje
- Monitoreá `loss_curve_` para ver la convergencia

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import numpy as np

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data, digits.target,
                                            test_size=0.2, random_state=42, stratify=digits.target)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# ✏️ Experimentá con estas configuraciones:
configs = [
    {'hidden_layer_sizes': (64,),         'alpha': 0.0001, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (128, 64),     'alpha': 0.0001, 'learning_rate_init': 0.001},
    # ✏️ Agregá 2 configs más variando alpha y la arquitectura:
]

print(f'{'Capas':20} {'alpha':>8} {'lr':>8} {'Acc Test':>10} {'Épocas':>8}')
print('-' * 57)

for cfg in configs:
    modelo = MLPClassifier(**cfg, max_iter=300, random_state=42, early_stopping=True)
    modelo.fit(X_tr_s, y_tr)
    acc    = modelo.score(X_te_s, y_te)
    epocas = len(modelo.loss_curve_)
    capas  = str(cfg['hidden_layer_sizes'])
    print(f'  {capas:18} {cfg["alpha"]:>8} {cfg["learning_rate_init"]:>8} {acc:>10.4f} {epocas:>8}')


<details>
<summary>👀 Ver solución — configuraciones adicionales</summary>

```python
configs = [
    {'hidden_layer_sizes': (64,),          'alpha': 0.0001, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (128, 64),      'alpha': 0.0001, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (256, 128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (128, 64),      'alpha': 0.01,   'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (128, 64),      'alpha': 0.0001, 'learning_rate_init': 0.01},
]
# Observá: más capas no siempre mejora, alpha alto regulariza más (menor acc pero más generalizable)
```
</details>

### 🚀 Ejercicio libre 1 — Red neuronal para clasificación de texto

Construí una red neuronal para clasificar reseñas de películas como positivas o negativas.

**Usando `sklearn.datasets.fetch_20newsgroups` o datos sintéticos:**

1. Convertí el texto a features numéricas con `TfidfVectorizer`
2. Dividí en train/test con `stratify`
3. Entrenás un `MLPClassifier` con al menos 2 capas ocultas
4. Comparalo contra `LogisticRegression` (baseline)
5. Analizá qué palabras son más determinantes para cada clase

**Si tenés TensorFlow:**
- Usá `keras.layers.Embedding` + LSTM para capturar contexto
- Compará con el MLP sin contexto

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 🚀 Clasificación de texto aquí:

# Datos sintéticos de reseñas:
resenas_pos = [
    'Excelente película, actuaciones increíbles y gran historia',
    'Me encantó, la dirección es magistral y el guión brillante',
    'Una obra maestra, la mejor del año sin duda',
    'Impresionante, emotiva y con un final perfecto',
] * 50
resenas_neg = [
    'Pésima película, aburrida y sin sentido alguno',
    'Una pérdida de tiempo, actuaciones terribles y guión malo',
    'Decepcionante, esperaba mucho más de este director',
    'Horrible, no la recomiendo para nada en absoluto',
] * 50

textos = resenas_pos + resenas_neg
etiquetas = [1]*200 + [0]*200

# Tu pipeline aquí:


---
## 📊 Proyecto 1: Análisis de Redes Sociales

### 📘 Descripción

Las **redes sociales** se modelan como grafos donde:
- **Nodos** = usuarios
- **Aristas** = conexiones (seguimientos, amistades, menciones)

#### Métricas de centralidad

| Métrica | Qué mide | Quién identifica |
|---------|----------|------------------|
| **Degree** | Cantidad de conexiones directas | Usuarios muy conectados |
| **Betweenness** | Cuántos caminos pasan por el nodo | Puentes entre comunidades |
| **PageRank** | Importancia basada en la calidad de conexiones | Influencers reales |

#### Detección de comunidades
```python
# Algoritmo de Louvain (maximiza modularidad)
# Implementación manual con BFS para encontrar componentes conexas
```

### 💡 Ejemplo resuelto 3 — Análisis completo de red social

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict, deque

# ✅ Análisis completo de red social implementado desde cero

class RedSocial:
    """Implementación de grafo para análisis de redes sociales."""

    def __init__(self, dirigido=False):
        self.grafo    = defaultdict(set)
        self.dirigido = dirigido
        self._nodos   = set()

    def agregar_arista(self, u, v):
        self.grafo[u].add(v)
        self._nodos.update([u, v])
        if not self.dirigido:
            self.grafo[v].add(u)

    @property
    def nodos(self):
        return self._nodos

    def grado(self, nodo):
        return len(self.grafo[nodo])

    def grados(self):
        return {n: self.grado(n) for n in self.nodos}

    def bfs_distancias(self, origen):
        """BFS para calcular distancias desde un nodo."""
        dist = {origen: 0}
        cola = deque([origen])
        while cola:
            nodo = cola.popleft()
            for vecino in self.grafo[nodo]:
                if vecino not in dist:
                    dist[vecino] = dist[nodo] + 1
                    cola.append(vecino)
        return dist

    def betweenness_centralidad(self):
        """Betweenness centrality: cuántos caminos cortos pasan por cada nodo."""
        entre = defaultdict(float)
        nodos = list(self.nodos)
        for origen in nodos:
            # BFS para encontrar caminos más cortos
            stack, pred, sigma, dist = [], defaultdict(list), defaultdict(float), {}
            sigma[origen] = 1.0
            dist[origen]  = 0
            cola = deque([origen])
            while cola:
                v = cola.popleft()
                stack.append(v)
                for w in self.grafo[v]:
                    if w not in dist:
                        cola.append(w)
                        dist[w] = dist[v] + 1
                    if dist[w] == dist[v] + 1:
                        sigma[w] += sigma[v]
                        pred[w].append(v)
            delta = defaultdict(float)
            while stack:
                w = stack.pop()
                for v in pred[w]:
                    delta[v] += (sigma[v]/sigma[w]) * (1 + delta[w])
                if w != origen:
                    entre[w] += delta[w]
        # Normalizar
        n = len(nodos)
        factor = 1 / ((n-1)*(n-2)) if n > 2 else 1
        if not self.dirigido:
            factor *= 2
        return {k: v*factor for k, v in entre.items()}

    def pagerank(self, damping=0.85, max_iter=100, tol=1e-6):
        """PageRank: importancia basada en la calidad de las conexiones."""
        nodos = list(self.nodos)
        n     = len(nodos)
        pr    = {nodo: 1/n for nodo in nodos}
        for _ in range(max_iter):
            pr_nuevo = {}
            for nodo in nodos:
                # Suma de PageRank de los que apuntan a este nodo
                suma = sum(pr[v]/len(self.grafo[v])
                           for v in nodos if nodo in self.grafo[v] and self.grafo[v])
                pr_nuevo[nodo] = (1-damping)/n + damping*suma
            # Verificar convergencia
            cambio = sum(abs(pr_nuevo[n]-pr[n]) for n in nodos)
            pr = pr_nuevo
            if cambio < tol:
                break
        return pr

    def detectar_comunidades_bfs(self):
        """Detecta componentes conexas con BFS."""
        visitados  = set()
        comunidades= []
        for nodo in self.nodos:
            if nodo not in visitados:
                comunidad = set()
                cola = deque([nodo])
                while cola:
                    n = cola.popleft()
                    if n not in visitados:
                        visitados.add(n)
                        comunidad.add(n)
                        cola.extend(self.grafo[n] - visitados)
                comunidades.append(comunidad)
        return sorted(comunidades, key=len, reverse=True)

    def amigos_en_comun(self, u, v):
        return self.grafo[u] & self.grafo[v]


# ── Crear red sintética ──
np.random.seed(42)
red = RedSocial(dirigido=False)

# Agregar influencers (muy conectados)
influencers = ['Ana','Luis','Marta','Pedro','Sofia']
usuarios    = [f'User_{i}' for i in range(1, 46)]
todos       = influencers + usuarios

# Cada influencer se conecta con muchos
for inf in influencers:
    seguidores = np.random.choice(usuarios, size=np.random.randint(15, 25), replace=False)
    for seg in seguidores:
        red.agregar_arista(inf, seg)

# Conexiones entre usuarios comunes
for _ in range(80):
    u, v = np.random.choice(usuarios, 2, replace=False)
    red.agregar_arista(u, v)

# Conexiones entre influencers
for i, inf1 in enumerate(influencers):
    for inf2 in influencers[i+1:]:
        if np.random.rand() > 0.3:
            red.agregar_arista(inf1, inf2)

print(f'Red: {len(red.nodos)} nodos, {sum(len(v) for v in red.grafo.values())//2} aristas')

# ── Análisis ──
grados = red.grados()
top_grado = sorted(grados.items(), key=lambda x: -x[1])[:8]
print('\n=== Top 8 por grado (conexiones directas) ===')
for nodo, g in top_grado:
    barra = '█' * g
    print(f'  {nodo:12}: {g:3}  {barra}')

print('\n=== PageRank — Influencers reales ===')
pr = red.pagerank()
top_pr = sorted(pr.items(), key=lambda x: -x[1])[:8]
for nodo, score in top_pr:
    barra = '█' * int(score * 1000)
    print(f'  {nodo:12}: {score:.4f}  {barra}')

print('\n=== Comunidades detectadas ===')
comunidades = red.detectar_comunidades_bfs()
for i, com in enumerate(comunidades[:3], 1):
    influencers_en_com = [n for n in com if n in influencers]
    print(f'  Comunidad {i}: {len(com)} nodos | Influencers: {influencers_en_com}')

print('\n=== Grados de separación (6 grados) ===')
pares = [('Ana', 'User_45'), ('Luis', 'User_30'), ('User_1', 'User_45')]
for u, v in pares:
    dist = red.bfs_distancias(u)
    d = dist.get(v, -1)
    print(f'  {u:12} → {v:12}: {d} saltos')

### ✏️ Ejercicio guiado 3 — Detección de influencers y puentes

Usá la red del ejemplo para encontrar los **puentes**: nodos cuya eliminación desconecta la red.

**Pistas:**
- Un nodo es **puente** si su betweenness centrality es significativamente mayor que el promedio
- Verificalo eliminando el nodo y contando las componentes conexas antes y después
- Los verdaderos influencers tienen alto PageRank Y alto degree

In [ ]:
# (Requiere red, red.betweenness_centralidad(), red.detectar_comunidades_bfs())

# ✏️ Calcular betweenness centrality
print('=== Betweenness Centrality — Nodos puente ===')
entre = red.betweenness_centralidad()
top_entre = sorted(entre.items(), key=lambda x: -x[1])[:5]
for nodo, score in top_entre:
    print(f'  {nodo:12}: {score:.4f}')

# ✏️ Combinar métricas para score de influencer
print('\n=== Score de influencer (grado + PageRank) ===')
pr = red.pagerank()
grados_n = red.grados()

# Normalizar ambas métricas al rango [0,1]
max_grado = max(grados_n.values())
max_pr    = max(pr.values())

scores = {
    nodo: 0.5 * (grados_n.get(nodo,0)/max_grado) + 0.5 * (pr.get(nodo,0)/max_pr)
    for nodo in red.nodos
}

top_scores = sorted(scores.items(), key=lambda x: -x[1])[:8]
print(f'  {'Nodo':12} {'Score':>8} {'Grado':>8} {'PageRank':>10}')
print('  ' + '-'*40)
for nodo, score in top_scores:
    g  = grados_n.get(nodo, 0)
    p  = pr.get(nodo, 0)
    print(f'  {nodo:12} {score:>8.4f} {g:>8} {p:>10.4f}')


### 🚀 Proyecto libre 3 — Tu análisis de red social

Construí un análisis completo de una red social real o sintética:

**Datos:**
- Descargá un dataset real de [SNAP (Stanford Network Analysis Project)](https://snap.stanford.edu/data/) 
- O generá una red más compleja con estructura de comunidades claras (Red de Barabási-Albert)

**Análisis requeridos:**
1. Distribución de grados (¿es una ley de potencias? → redes reales)
2. Top 10 influencers por PageRank, grado y betweenness
3. Detección de al menos 3 comunidades
4. Distancia promedio entre nodos (coeficiente de mundo pequeño)
5. Visualización de la red con tamaño de nodo proporcional al PageRank
6. ¿Qué nodo, si se elimina, desconecta más la red?

In [ ]:
import numpy as np
from collections import defaultdict, deque

# 🚀 Tu análisis de red social aquí:

# Podés usar la clase RedSocial del ejemplo anterior
# o implementar tu propia versión extendida


---
## 🏠 Proyecto 2: Predicción de Precios de Bienes Raíces

### 📘 Descripción

Este proyecto integra el ciclo completo de ciencia de datos:
**datos → limpieza → EDA → features → modelo → evaluación → interpretación**

Vas a trabajar con un dataset sintético realista que simula el mercado inmobiliario argentino.

### 💡 Ejemplo resuelto 4 — Pipeline completo de predicción de precios

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')

# ✅ Pipeline completo de predicción de precios

np.random.seed(42)
n = 2000

# Dataset enriquecido
barrios  = ['Palermo','Belgrano','Recoleta','San Telmo','Almagro',
             'Caballito','Villa Urquiza','Flores','Boedo','Colegiales']
tipos    = ['Departamento','Casa','PH','Duplex']
estados  = ['Excelente','Muy bueno','Bueno','Regular']

df = pd.DataFrame({
    'barrio':      np.random.choice(barrios, n),
    'tipo':        np.random.choice(tipos, n, p=[0.55,0.2,0.15,0.1]),
    'estado':      np.random.choice(estados, n, p=[0.2,0.35,0.3,0.15]),
    'superficie':  np.random.normal(72, 28, n).clip(20, 250),
    'ambientes':   np.random.choice([1,2,3,4,5], n, p=[0.08,0.32,0.38,0.17,0.05]),
    'banos':       np.random.choice([1,2,3], n, p=[0.55,0.35,0.10]),
    'antiguedad':  np.random.randint(0, 80, n),
    'piso':        np.random.randint(1, 22, n),
    'cochera':     np.random.choice([0,1], n, p=[0.55,0.45]),
    'amenities':   np.random.randint(0, 6, n),
    'distancia_subte': np.random.exponential(4, n).clip(0.1, 15).round(1),
})

# Precios con lógica realista
mult_barrio = {'Palermo':1.35,'Belgrano':1.25,'Recoleta':1.4,'San Telmo':1.05,
               'Almagro':1.0,'Caballito':1.08,'Villa Urquiza':1.12,
               'Flores':0.88,'Boedo':0.92,'Colegiales':1.18}
mult_tipo   = {'Departamento':1.0,'Casa':1.15,'PH':0.9,'Duplex':1.2}
mult_estado = {'Excelente':1.15,'Muy bueno':1.05,'Bueno':1.0,'Regular':0.85}

df['precio'] = (
    1150 * df['superficie']
    + 5000 * df['ambientes']
    + 8000 * df['banos']
    + 15000 * df['cochera']
    + 3000  * df['amenities']
    - 220   * df['antiguedad']
    + 400   * df['piso']
    - 2000  * df['distancia_subte']
    + np.random.normal(0, 12000, n)
) * df['barrio'].map(mult_barrio) * df['tipo'].map(mult_tipo) * df['estado'].map(mult_estado)
df['precio'] = df['precio'].clip(40000, 800000).round(-3)

print(f'Dataset: {len(df)} propiedades')
print(f'Precio — media: ${df["precio"].mean():,.0f} | mediana: ${df["precio"].median():,.0f}')
print(f'Rango: ${df["precio"].min():,.0f} — ${df["precio"].max():,.0f}')

# ── Feature Engineering ──
df['sup_por_amb']   = df['superficie'] / df['ambientes']
df['precio_m2_est'] = df['precio'] / df['superficie']  # solo para EDA, no para el modelo
df['es_nuevo']      = (df['antiguedad'] <= 3).astype(int)
df['piso_alto']     = (df['piso'] >= 10).astype(int)
df['premium']       = ((df['cochera'] == 1) & (df['amenities'] >= 3)).astype(int)

# ── Preparar features ──
num_cols = ['superficie','ambientes','banos','antiguedad','piso','cochera',
            'amenities','distancia_subte','sup_por_amb','es_nuevo','piso_alto','premium']
cat_cols = ['barrio','tipo','estado']
X = df[num_cols + cat_cols]
y = df['precio']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
])

modelos = [
    ('Ridge',              Ridge(alpha=1.0)),
    ('Random Forest',      RandomForestRegressor(n_estimators=200, random_state=42)),
    ('Gradient Boosting',  GradientBoostingRegressor(n_estimators=200, random_state=42)),
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(f'\n{'Modelo':22} {'MAE':>12} {'R² Test':>10} {'R² CV':>10}')
print('-' * 56)

mejor_r2, mejor_pipe = 0, None
for nombre, modelo in modelos:
    pipe = Pipeline([('prep', prep), ('modelo', modelo)])
    pipe.fit(X_tr, y_tr)
    pred    = pipe.predict(X_te)
    mae     = mean_absolute_error(y_te, pred)
    r2_test = r2_score(y_te, pred)
    r2_cv   = cross_val_score(pipe, X_tr, y_tr, cv=kf, scoring='r2').mean()
    print(f'  {nombre:20} ${mae:>10,.0f} {r2_test:>10.4f} {r2_cv:>10.4f}')
    if r2_test > mejor_r2:
        mejor_r2, mejor_pipe = r2_test, pipe

# Análisis de errores por segmento
pred_mejor = mejor_pipe.predict(X_te)
df_err = X_te.copy()
df_err['real']      = y_te.values
df_err['pred']      = pred_mejor
df_err['error_pct'] = abs(df_err['real'] - df_err['pred']) / df_err['real'] * 100

print('\n=== Error % por barrio ===')
err_barrio = df_err.groupby('barrio')['error_pct'].agg(['mean','count'])
err_barrio.columns = ['error_pct_avg','n']
print(err_barrio.sort_values('error_pct_avg').round(1).to_string())

# Feature importance del Random Forest
rf_pipe = [p for n, p in [(nombre,pipe) for nombre, _ in modelos]
           if 'Forest' in n]
if rf_pipe:
    rf_pipe2 = Pipeline([('prep',prep),
        ('modelo',RandomForestRegressor(n_estimators=200,random_state=42))])
    rf_pipe2.fit(X_tr, y_tr)
    feat_names = num_cols + list(rf_pipe2.named_steps['prep']
        .named_transformers_['cat'].get_feature_names_out(cat_cols))
    importancias = pd.Series(rf_pipe2.named_steps['modelo'].feature_importances_,
                             index=feat_names).sort_values(ascending=False)
    print('\n=== Feature Importance (Random Forest) ===')
    for feat, imp in importancias.head(10).items():
        barra = '█' * int(imp * 100)
        print(f'  {feat:25}: {imp:.4f}  {barra}')

### 🚀 Proyecto libre 4 — Análisis inmobiliario completo

Extendé el modelo de bienes raíces con estos análisis adicionales:

1. **Mapa de calor de precios por barrio** — usá `seaborn.heatmap` o `matplotlib.imshow`
2. **Análisis de residuos** — graficá `(y_real - y_pred)` vs `y_pred` para detectar patrones
3. **Predictor interactivo** — función `estimar_precio(superficie, ambientes, barrio, ...)` que devuelve el precio estimado con intervalo de confianza
4. **Segmentación RFM aplicada a propiedades** — clasificá propiedades en 'Oportunidad', 'Premium', 'Sobrevalorada' basándote en precio/m² vs promedio del barrio
5. **Reporte automático** — generá un informe de texto con los hallazgos más importantes del modelo

In [ ]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')

# 🚀 Tu análisis inmobiliario extendido aquí:


---
## 🏥 Proyecto 3: Clasificación de Enfermedades Médicas

### 📘 Descripción

En medicina, los modelos de ML deben ser **interpretables** además de precisos.  
Un médico necesita entender POR QUÉ el modelo predice una enfermedad, no solo el resultado.

#### Consideraciones especiales en ML médico

| Aspecto | Consideración |
|---------|---------------|
| **Recall vs Precision** | En diagnóstico, el recall (no perder enfermos) suele importar más |
| **Interpretabilidad** | Feature importance, SHAP values para explicar predicciones |
| **Desbalance** | Las enfermedades raras tienen muy pocos casos positivos |
| **Validación** | Cross-validation estratificada es crucial |
| **Umbral** | Ajustar según el costo de falsos negativos vs falsos positivos |

### 💡 Ejemplo resuelto 5 — Clasificación médica con interpretabilidad

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                              precision_recall_curve, confusion_matrix)
import warnings; warnings.filterwarnings('ignore')

# ✅ Dataset sintético de diabetes (inspirado en Pima Indians Diabetes)

np.random.seed(42)
n = 1500

df_med = pd.DataFrame({
    'glucosa':          np.random.normal(120, 32, n).clip(44, 280),
    'presion_arterial': np.random.normal(72, 12, n).clip(24, 122),
    'imc':              np.random.normal(32, 8, n).clip(14, 67),
    'edad':             np.random.randint(21, 81, n),
    'insulina':         np.random.exponential(80, n).clip(0, 850),
    'grosor_piel':      np.random.normal(29, 11, n).clip(0, 99),
    'dpf':              np.random.exponential(0.47, n).clip(0.07, 2.42),  # diabetes pedigree function
    'embarazos':        np.random.choice(range(18), n, p=[0.25]+[0.05]*15),
})

# Probabilidad de diabetes basada en factores de riesgo reales
prob = (
    -6.0
    + 0.035 * df_med['glucosa']
    + 0.015 * df_med['imc']
    + 0.025 * df_med['edad']
    + 0.8   * df_med['dpf']
    + 0.005 * df_med['insulina']
    + 0.02  * df_med['embarazos']
)
prob_sigmoid = 1 / (1 + np.exp(-prob))
df_med['diabetes'] = np.random.binomial(1, prob_sigmoid)

print(f'Dataset médico: {len(df_med)} pacientes')
print(f'Prevalencia diabetes: {df_med["diabetes"].mean()*100:.1f}%')
print(f'\nEstadísticas por condición:')
print(df_med.groupby('diabetes')[['glucosa','imc','edad']].mean().round(1))

# Features y target
features = ['glucosa','presion_arterial','imc','edad','insulina','grosor_piel','dpf','embarazos']
X = df_med[features]
y = df_med['diabetes']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos_med = [
    ('Logistic Regression', LogisticRegression(class_weight='balanced', max_iter=1000)),
    ('Random Forest',       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)),
    ('Gradient Boosting',   GradientBoostingClassifier(n_estimators=200, random_state=42)),
]

print(f'\n{'Modelo':22} {'AUC-ROC':>10} {'Recall (enf)':>14} {'F1 macro':>10}')
print('-' * 58)

mejor_auc, mejor_pipe_med = 0, None
for nombre, modelo in modelos_med:
    pipe = Pipeline([('scaler', StandardScaler()), ('modelo', modelo)])
    pipe.fit(X_tr, y_tr)
    pred       = pipe.predict(X_te)
    pred_proba = pipe.predict_proba(X_te)[:, 1]
    auc        = roc_auc_score(y_te, pred_proba)
    from sklearn.metrics import recall_score, f1_score
    recall_enf = recall_score(y_te, pred)
    f1         = f1_score(y_te, pred, average='macro')
    print(f'  {nombre:20} {auc:>10.4f} {recall_enf:>14.4f} {f1:>10.4f}')
    if auc > mejor_auc:
        mejor_auc, mejor_pipe_med = auc, pipe

# Reporte del mejor modelo
pred_mejor = mejor_pipe_med.predict(X_te)
print(f'\n=== Reporte detallado ===')
print(classification_report(y_te, pred_mejor, target_names=['Sano','Diabético']))

# Análisis de umbral — en medicina el recall importa más
proba_mejor = mejor_pipe_med.predict_proba(X_te)[:,1]
precisions, recalls, thresholds = precision_recall_curve(y_te, proba_mejor)
f1_scores = 2*precisions*recalls/(precisions+recalls+1e-8)
umbral_opt = thresholds[np.argmax(f1_scores[:-1])]

print(f'\n=== Análisis de umbral ===')
print(f'  Umbral default (0.5):')
cm = confusion_matrix(y_te, (proba_mejor >= 0.5).astype(int))
tn,fp,fn,tp = cm.ravel()
print(f'    TN={tn} FP={fp} FN={fn} TP={tp}')
print(f'    Enfermos no detectados (FN): {fn}')

print(f'  Umbral optimizado ({umbral_opt:.2f}):')
cm2 = confusion_matrix(y_te, (proba_mejor >= umbral_opt).astype(int))
tn2,fp2,fn2,tp2 = cm2.ravel()
print(f'    TN={tn2} FP={fp2} FN={fn2} TP={tp2}')
print(f'    Enfermos no detectados (FN): {fn2}')

# Feature importance
rf_med = Pipeline([('scaler',StandardScaler()),
    ('modelo',RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=42))])
rf_med.fit(X_tr, y_tr)
importancias = pd.Series(rf_med.named_steps['modelo'].feature_importances_, index=features)
print('\n=== Variables más importantes para el diagnóstico ===')
for feat, imp in importancias.sort_values(ascending=False).items():
    barra = '█' * int(imp * 100)
    print(f'  {feat:20}: {imp:.4f}  {barra}')

### 🚀 Proyecto libre 5 — Pipeline médico completo con interpretabilidad

Extendé el modelo médico con análisis de interpretabilidad:

1. **Análisis por subgrupos** — ¿el modelo funciona igual de bien en jóvenes vs mayores? ¿Hombres vs mujeres?
2. **Casos extremos** — identifica los 10 casos donde el modelo más se equivoca y analizalos
3. **Reglas de decisión simples** — extraé las reglas del árbol más representativo del Random Forest
4. **Perfil de riesgo** — función `evaluar_paciente(datos_paciente)` que retorna:
   - Probabilidad de diabetes
   - Nivel de riesgo ('Bajo'/'Medio'/'Alto')
   - Los 3 factores que más contribuyen al riesgo del paciente
   - Recomendaciones basadas en las variables más importantes
5. **Validación clínica** — ¿cuántos diagnósticos incorrectos tendría el modelo en 10.000 pacientes?

In [ ]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings('ignore')

# 🚀 Tu pipeline médico con interpretabilidad aquí:

# Función evaluar_paciente:
def evaluar_paciente(modelo, features, datos_paciente):
    """
    Evalúa el riesgo de diabetes de un paciente.
    datos_paciente: dict con las features del paciente
    """
    pass

# Test:
# resultado = evaluar_paciente(rf_med, features, {
#     'glucosa': 148, 'presion_arterial': 72, 'imc': 33.6,
#     'edad': 50, 'insulina': 0, 'grosor_piel': 35,
#     'dpf': 0.627, 'embarazos': 6
# })


---
## 🎓 Cierre del Curso — Semana 23-24

### ✅ Lo que aprendiste en esta semana

| Tema | Conceptos clave |
|------|-----------------|
| TensorFlow/Keras | Redes neuronales, capas, activaciones, entrenamiento |
| MLPClassifier | Arquitecturas, regularización, early stopping |
| Redes sociales | Grafos, PageRank, betweenness, comunidades |
| Bienes raíces | Pipeline completo, feature importance, análisis de errores |
| ML médico | Interpretabilidad, umbral óptimo, consideraciones éticas |

---

## 🏆 Resumen del Curso Completo — 24 Semanas

### Lo que construiste durante el curso

| Semana | Tema principal | Habilidades adquiridas |
|--------|---------------|------------------------|
| 1-2 | Fundamentos Python + SQL | Sintaxis, estructuras, SQLite, NoSQL |
| 3-4 | SQL Avanzado | JOINs, índices, normalización, CTEs |
| 5-6 | Backend + APIs | Flask, requests, pandas, NumPy |
| 7-8 | Algoritmos y estructuras | Big O, búsqueda, ordenamiento, grafos |
| 9-10 | Proyectos prácticos | App web, pipeline datos, API REST |
| 11-12 | Frameworks avanzados | Blueprints, SQLAlchemy ORM, APIs externas |
| 13-14 | Testing y BD avanzada | unittest, pytest, coverage, transacciones |
| 15-16 | Seguridad y rendimiento | SQL injection, RBAC, caché LRU, índices |
| 17-18 | Patrones + Data Science | MVC/Factory/Observer, EDA, ML básico |
| 19-20 | Visualización | Pandas avanzado, NumPy, Matplotlib, Seaborn |
| 21-22 | ML + SQL window | Regresión/Clasificación, Window Functions |
| 23-24 | Deep Learning + Proyectos | TensorFlow, redes sociales, ML médico |

### 🚀 Próximos pasos recomendados

**Para especializarte en Data Science:**
- Deep Learning avanzado: CNNs, RNNs, Transformers
- NLP: procesamiento de lenguaje natural con spaCy y HuggingFace
- MLOps: despliegue de modelos con Docker, FastAPI y MLflow

**Para especializarte en Backend:**
- Django REST Framework y FastAPI en profundidad
- Bases de datos distribuidas: PostgreSQL, Redis, MongoDB
- Arquitectura de microservicios y contenedores

**Para especializarte en Business Analytics:**
- Power BI y Tableau para visualizaciones ejecutivas
- SQL analítico: Snowflake, BigQuery, Redshift
- A/B Testing y estadística para negocios

---

### 📚 Recursos finales recomendados
- [Fast.ai — Practical Deep Learning](https://course.fast.ai/) — el mejor curso de DL práctico
- [Kaggle Learn](https://www.kaggle.com/learn) — micro-cursos gratuitos de ML
- [Towards Data Science](https://towardsdatascience.com/) — artículos de la comunidad
- [The Pragmatic Programmer](https://pragprog.com/titles/tpp20/) — libro esencial para todo dev
- [Made With ML](https://madewithml.com/) — MLOps y ML en producción

---

## 💬 Reflexión final

> *"La ciencia de datos no es sobre los modelos. Es sobre las preguntas que hacés,  
> los datos que recopilás, y las decisiones que ayudás a tomar."*

Felicitaciones por completar las 24 semanas. 🎉
El conocimiento técnico es solo la mitad del camino —  
la otra mitad es la curiosidad para seguir aprendiendo.

## 📝 Mi portafolio — Proyectos completados

| Proyecto | Descripción | Tecnologías | Link |
|----------|-------------|-------------|------|
| App web | | Flask, SQLAlchemy | |
| Pipeline datos | | Pandas, NumPy | |
| API REST | | Flask, SQLite | |
| Análisis redes | | Python, grafos | |
| Predicción precios | | scikit-learn, RF | |
| ML médico | | scikit-learn, RF | |

*✍️ Completá esta tabla con tus proyectos — doble click para editar*